# PRT661 Data Science Practice — Fire2Air Darwin
## Assessment 2 technical implementation notebook

**Project:** Explainable next-day smoke and PM₂.₅ forecasting for Greater Darwin  
**Theme:** Theme 2 — Predictive Analytics and Forecasting  
**Stations:** Palmerston, Winnellie, Stokes Hill  
**Prediction horizon:** day *t* → day *t+1*

This notebook is designed to run in **VS Code / Jupyter** with the notebook and raw CSV files in the same project folder (or with the notebook one folder below the data). It implements the technical evidence requested in the Assessment 2 requirements and stays aligned with the Assessment 1 proposal.

### Three prediction tasks
1. **Classification:** whether tomorrow's 24-hour mean PM₂.₅ exceeds **25 µg/m³**.
2. **Regression:** tomorrow's 24-hour mean PM₂.₅ concentration.
3. **Count prediction:** number of tomorrow's hours with PM₂.₅ above **25 µg/m³** (0–24).

### Leakage-safe evaluation
- **Training:** 2018–2022 where both datasets have coverage.
- **Validation:** 2023.
- **Final untouched test:** 2024.
- Features for day *t+1* use only information available on or before day *t*.

> **Important coverage safeguard:** the uploaded S-NPP files currently visible to this notebook may begin in 2019. The code dynamically audits the actual FIRMS date coverage. A year with no supplied FIRMS archive is **not** silently interpreted as a zero-fire year; it is excluded from modelling until the missing archive is added.

### Decision-support statement
The model is a research/decision-support prototype. It is **not** an official NT EPA warning and is **not medical advice**. SHAP/feature importance explains model behaviour; it does not prove that a particular fire caused a pollution event.

## 0. Environment and reproducibility

Recommended packages:

```text
pandas
numpy
matplotlib
scikit-learn
xgboost
shap
joblib
duckdb            # optional storage evidence
```

The notebook does not modify raw CSV files. All generated evidence is written under `outputs_prt661/`.

In [ ]:
from pathlib import Path
import json
import math
import platform
import re
import sys
import time
import warnings
from copy import deepcopy

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, PoissonRegressor, Ridge, TweedieRegressor
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    mean_absolute_error,
    mean_poisson_deviance,
    mean_squared_error,
    precision_recall_curve,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 661
np.random.seed(RANDOM_STATE)

try:
    from xgboost import XGBClassifier, XGBRegressor
    HAS_XGBOOST = True
except Exception as exc:
    HAS_XGBOOST = False
    print(f"XGBoost unavailable: {exc}")

try:
    import shap
    HAS_SHAP = True
except Exception as exc:
    HAS_SHAP = False
    print(f"SHAP unavailable: {exc}")

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

In [ ]:
# -------------------------
# Project configuration
# -------------------------
EPA_PATTERN = "ntepa_greater_darwin_hourly_2018_2024*.csv"
FIRMS_PATTERN = "fire_archive_SV-C2_*.csv"

PROJECT_YEARS = list(range(2018, 2025))
TRAIN_END_YEAR = 2022
VALIDATION_YEAR = 2023
TEST_YEAR = 2024

PM25_THRESHOLD = 25.0
MIN_VALID_PM25_HOURS_FOR_DAILY_MEAN = 18
MIN_VALID_PM25_HOURS_FOR_COUNT = 24

# Reference point used only for a fast pre-filter before station-specific distances.
DARWIN_REFERENCE_LAT = -12.4634
DARWIN_REFERENCE_LON = 130.8456
DARWIN_PREFILTER_KM = 520.0
MAX_FIRE_DISTANCE_KM = 500.0
DISTANCE_BANDS_KM = (50, 100, 250, 500)

# Conservative engineering bounds used to remove obvious sensor/sentinel faults.
# High positive PM values are deliberately retained because real smoke episodes can be extreme.
PLAUSIBLE_RANGES = {
    "relative_humidity_pct": (0.0, 100.0),
    "air_temperature_c": (-20.0, 60.0),
    "wind_speed_m_s": (0.0, 80.0),
    "wind_direction_deg": (0.0, 360.0),
    "air_pressure_hpa": (850.0, 1100.0),
}
MAX_PLAUSIBLE_HOURLY_RAIN_MM = 300.0

def locate_project_root():
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for candidate in candidates:
        if list(candidate.glob(EPA_PATTERN)):
            return candidate
    raise FileNotFoundError(
        "Could not find the NT EPA CSV. Put this notebook in the same folder as the datasets "
        "or one folder below them."
    )

PROJECT_ROOT = locate_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "outputs_prt661"
FIGURE_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR = OUTPUT_ROOT / "tables"
MODEL_DIR = OUTPUT_ROOT / "models"
PROCESSED_DIR = OUTPUT_ROOT / "processed"

for folder in [OUTPUT_ROOT, FIGURE_DIR, TABLE_DIR, MODEL_DIR, PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

epa_candidates = sorted(PROJECT_ROOT.glob(EPA_PATTERN))
if not epa_candidates:
    raise FileNotFoundError(f"No file matching {EPA_PATTERN}")

EPA_PATH = epa_candidates[0]
FIRMS_PATHS = sorted(PROJECT_ROOT.glob(FIRMS_PATTERN))

if not FIRMS_PATHS:
    raise FileNotFoundError(f"No files matching {FIRMS_PATTERN}")

print("Project root:", PROJECT_ROOT)
print("NT EPA:", EPA_PATH.name)
print(f"FIRMS files found: {len(FIRMS_PATHS)}")
for p in FIRMS_PATHS:
    print(" -", p.name)

## 2. Reusable helper functions
These functions keep the notebook reproducible and make validation logic explicit rather than hiding it inside one long script.

In [ ]:
EARTH_RADIUS_KM = 6371.0088

def haversine_km(lat, lon, ref_lat, ref_lon):
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    lat1 = np.radians(ref_lat)
    lon1 = np.radians(ref_lon)
    lat2 = np.radians(lat)
    lon2 = np.radians(lon)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    a = np.clip(a, 0.0, 1.0)
    return EARTH_RADIUS_KM * 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))

def initial_bearing_deg(origin_lat, origin_lon, dest_lat, dest_lon):
    lat1 = np.radians(origin_lat)
    lat2 = np.radians(np.asarray(dest_lat, dtype=float))
    dlon = np.radians(np.asarray(dest_lon, dtype=float) - origin_lon)
    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(y, x)) + 360.0) % 360.0

def circular_mean_deg(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return np.nan
    radians = np.radians(s.to_numpy())
    sin_mean = np.mean(np.sin(radians))
    cos_mean = np.mean(np.cos(radians))
    if np.isclose(sin_mean, 0) and np.isclose(cos_mean, 0):
        return np.nan
    return (np.degrees(np.arctan2(sin_mean, cos_mean)) + 360.0) % 360.0

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def safe_roc_auc(y_true, y_prob):
    return float(roc_auc_score(y_true, y_prob)) if pd.Series(y_true).nunique() > 1 else np.nan

def parse_year_from_name(path):
    match = re.search(r"(20\d{2})", path.stem)
    return int(match.group(1)) if match else np.nan

def save_table(df, filename, index=False):
    path = TABLE_DIR / filename
    df.to_csv(path, index=index)
    print("Saved:", path.relative_to(PROJECT_ROOT))
    return path

def to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (pd.Timestamp,)):
        return obj.isoformat()
    if pd.isna(obj) if not isinstance(obj, (dict, list, tuple)) else False:
        return None
    return obj

# Part A — NT EPA Air Quality and Meteorology

## 3. Acquisition, schema, coverage and raw quality audit
The raw dataset is inspected before any values are changed. Evidence is saved to CSV for the Assessment 2 report.

In [ ]:
epa_raw = pd.read_csv(EPA_PATH)
epa_raw["datetime_local"] = pd.to_datetime(epa_raw["datetime_local"], errors="coerce")

EXPECTED_EPA_COLUMNS = {
    "datetime_local", "station", "latitude", "longitude",
    "pm25_ug_m3", "pm10_ug_m3", "relative_humidity_pct",
    "air_temperature_c", "wind_speed_m_s", "wind_direction_deg",
    "air_pressure_hpa", "rainfall_mm",
}
missing_cols = EXPECTED_EPA_COLUMNS.difference(epa_raw.columns)
if missing_cols:
    raise ValueError(f"NT EPA file is missing required columns: {sorted(missing_cols)}")

epa_overview = pd.DataFrame({
    "metric": [
        "rows", "columns", "date_min", "date_max", "stations",
        "full_row_duplicates", "station_timestamp_duplicates"
    ],
    "value": [
        len(epa_raw), epa_raw.shape[1],
        epa_raw["datetime_local"].min(),
        epa_raw["datetime_local"].max(),
        ", ".join(sorted(epa_raw["station"].dropna().astype(str).unique())),
        int(epa_raw.duplicated().sum()),
        int(epa_raw.duplicated(["station", "datetime_local"]).sum()),
    ],
})
display(epa_overview)
save_table(epa_overview, "epa_raw_overview.csv")

In [ ]:
QUALITY_COLUMNS = [
    "pm25_ug_m3", "pm10_ug_m3", "relative_humidity_pct",
    "air_temperature_c", "wind_speed_m_s", "wind_direction_deg",
    "air_pressure_hpa", "rainfall_mm",
]

epa_raw["year"] = epa_raw["datetime_local"].dt.year

missing_pct = (
    epa_raw.groupby(["year", "station"])[QUALITY_COLUMNS]
    .agg(lambda s: s.isna().mean() * 100)
    .round(2)
)
display(missing_pct)
missing_pct.reset_index().to_csv(TABLE_DIR / "epa_missing_pct_year_station_variable.csv", index=False)

# Compact overall missingness table
overall_missing = pd.DataFrame({
    "missing_count": epa_raw[QUALITY_COLUMNS].isna().sum(),
    "missing_pct": (epa_raw[QUALITY_COLUMNS].isna().mean() * 100).round(2),
}).sort_values("missing_pct", ascending=False)
display(overall_missing)
save_table(overall_missing.reset_index(names="variable"), "epa_overall_missingness.csv")

In [ ]:
# Raw range / anomaly evidence. IQR extremes are reported, NOT automatically deleted.
raw_numeric_summary = epa_raw[QUALITY_COLUMNS].describe(
    percentiles=[0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
).T
display(raw_numeric_summary)
save_table(raw_numeric_summary.reset_index(names="variable"), "epa_raw_numeric_summary.csv")

iqr_rows = []
for col in QUALITY_COLUMNS:
    s = pd.to_numeric(epa_raw[col], errors="coerce").dropna()
    if s.empty:
        continue
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    iqr_rows.append({
        "variable": col,
        "q1": q1, "q3": q3, "iqr": iqr,
        "lower_iqr_fence": lower, "upper_iqr_fence": upper,
        "n_outside_iqr_fence": int(((s < lower) | (s > upper)).sum()),
    })
iqr_report = pd.DataFrame(iqr_rows)
display(iqr_report)
save_table(iqr_report, "epa_iqr_outlier_audit_no_automatic_deletion.csv")

## 4. Cleaning and unit harmonisation

The code applies transparent, auditable rules:

- negative PM₂.₅/PM₁₀ values → missing;
- relative humidity outside 0–100% → missing;
- obvious temperature, wind and pressure sentinel/invalid values → missing;
- station pressure values with a median around 1.0 are interpreted as **bar-like** and converted to hPa (`×1000`) before range validation;
- rainfall is profiled by station. A series that behaves like a cumulative counter is differenced only across consecutive hourly timestamps. Counter resets/gaps are treated as unknown rather than as rainfall;
- IQR/statistical extremes are **not** automatically deleted, because high PM/FRP values may be genuine fire/smoke events.

These engineering choices are written to an audit table so they can be justified in the report.

In [ ]:
epa = epa_raw.drop(columns=["year"]).copy()
cleaning_log = []

def log_change(variable, rule, before_nonmissing, after_nonmissing):
    cleaning_log.append({
        "variable": variable,
        "rule": rule,
        "values_set_or_changed": int(before_nonmissing - after_nonmissing)
    })

# PM: remove physically impossible negative concentrations, retain high positives.
for col in ["pm25_ug_m3", "pm10_ug_m3"]:
    before = epa[col].notna().sum()
    epa.loc[pd.to_numeric(epa[col], errors="coerce") < 0, col] = np.nan
    after = epa[col].notna().sum()
    log_change(col, "negative values -> NaN; high positive values retained", before, after)

# Pressure unit audit and harmonisation.
pressure_profile = (
    epa.groupby("station")["air_pressure_hpa"]
    .agg(["count", "median", "min", "max"])
    .reset_index()
)
pressure_profile["conversion_factor"] = np.where(
    pressure_profile["median"].between(0.8, 1.2, inclusive="both"), 1000.0, 1.0
)
display(pressure_profile)
save_table(pressure_profile, "epa_pressure_unit_audit.csv")

for row in pressure_profile.itertuples(index=False):
    if row.conversion_factor != 1.0:
        mask = epa["station"].eq(row.station) & epa["air_pressure_hpa"].notna()
        epa.loc[mask, "air_pressure_hpa"] = (
            epa.loc[mask, "air_pressure_hpa"] * row.conversion_factor
        )
        cleaning_log.append({
            "variable": "air_pressure_hpa",
            "rule": f"{row.station}: median suggested bar-like scale; multiplied by 1000 to hPa",
            "values_set_or_changed": int(mask.sum()),
        })

# Apply broad physical/sensor plausibility rules.
for col, (low, high) in PLAUSIBLE_RANGES.items():
    before = epa[col].notna().sum()
    if col == "wind_direction_deg":
        valid = epa[col].between(low, high, inclusive="left")
    else:
        valid = epa[col].between(low, high, inclusive="both")
    epa.loc[epa[col].notna() & ~valid, col] = np.nan
    after = epa[col].notna().sum()
    log_change(col, f"outside [{low}, {high}] -> NaN", before, after)

# Rainfall behaviour audit.
rain_profile_rows = []
for station, group in epa.sort_values("datetime_local").groupby("station"):
    s = group["rainfall_mm"]
    nonmissing = s.dropna()
    rain_profile_rows.append({
        "station": station,
        "nonmissing_pct": 100 * s.notna().mean(),
        "median_raw": nonmissing.median() if not nonmissing.empty else np.nan,
        "p95_raw": nonmissing.quantile(0.95) if not nonmissing.empty else np.nan,
        "zero_fraction_raw": (nonmissing.eq(0).mean() if not nonmissing.empty else np.nan),
        "n_unique_raw": nonmissing.nunique(),
    })

rain_profile = pd.DataFrame(rain_profile_rows)
# Data-driven flag: high baseline + almost never zero is characteristic of a cumulative counter.
rain_profile["cumulative_counter_flag"] = (
    (rain_profile["median_raw"] > 100) &
    (rain_profile["zero_fraction_raw"].fillna(1.0) < 0.20)
)
display(rain_profile)
save_table(rain_profile, "epa_rainfall_station_behaviour_audit.csv")

In [ ]:
# Create a harmonised hourly rainfall feature without overwriting the raw rainfall field.
epa = epa.sort_values(["station", "datetime_local"]).copy()
epa["rainfall_hourly_mm"] = np.nan

cumulative_stations = set(
    rain_profile.loc[rain_profile["cumulative_counter_flag"], "station"]
)

for station, idx in epa.groupby("station").groups.items():
    g = epa.loc[idx].sort_values("datetime_local")
    raw_rain = pd.to_numeric(g["rainfall_mm"], errors="coerce")

    if station in cumulative_stations:
        diffs = raw_rain.diff()
        hour_gap = g["datetime_local"].diff().dt.total_seconds().div(3600)
        derived = diffs.where(hour_gap.eq(1))
        # Negative changes are counter resets or inconsistent readings: do not count as rainfall.
        derived = derived.where(derived >= 0)
        derived = derived.where(derived <= MAX_PLAUSIBLE_HOURLY_RAIN_MM)
        epa.loc[g.index, "rainfall_hourly_mm"] = derived
    else:
        direct = raw_rain.where(raw_rain.between(0, MAX_PLAUSIBLE_HOURLY_RAIN_MM))
        epa.loc[g.index, "rainfall_hourly_mm"] = direct

cleaning_log_df = pd.DataFrame(cleaning_log)
display(cleaning_log_df)
save_table(cleaning_log_df, "epa_cleaning_log.csv")

cleaned_missing = pd.DataFrame({
    "missing_count": epa[QUALITY_COLUMNS + ["rainfall_hourly_mm"]].isna().sum(),
    "missing_pct": (epa[QUALITY_COLUMNS + ["rainfall_hourly_mm"]].isna().mean() * 100).round(2),
}).sort_values("missing_pct", ascending=False)
display(cleaned_missing)
save_table(cleaned_missing.reset_index(names="variable"), "epa_cleaned_missingness.csv")

## 5. Daily station aggregation

For daily mean PM₂.₅, the notebook requires at least **18 valid hourly observations** (75% coverage).  
For the 0–24 elevated-hours target, it requires **all 24 PM₂.₅ hours**, because a partial day can systematically undercount exposure duration.

Wind direction is aggregated with sine/cosine components rather than an arithmetic mean, avoiding the 359°/1° discontinuity.

In [ ]:
epa["date"] = epa["datetime_local"].dt.normalize()
epa["wind_sin_hour"] = np.sin(np.radians(epa["wind_direction_deg"]))
epa["wind_cos_hour"] = np.cos(np.radians(epa["wind_direction_deg"]))

daily_air = (
    epa.groupby(["station", "date"], as_index=False)
    .agg(
        latitude=("latitude", "median"),
        longitude=("longitude", "median"),
        pm25_mean=("pm25_ug_m3", "mean"),
        pm25_max=("pm25_ug_m3", "max"),
        pm25_valid_hours=("pm25_ug_m3", "count"),
        pm10_mean=("pm10_ug_m3", "mean"),
        pm10_valid_hours=("pm10_ug_m3", "count"),
        relative_humidity_mean=("relative_humidity_pct", "mean"),
        air_temperature_mean=("air_temperature_c", "mean"),
        wind_speed_mean=("wind_speed_m_s", "mean"),
        wind_sin=("wind_sin_hour", "mean"),
        wind_cos=("wind_cos_hour", "mean"),
        air_pressure_mean=("air_pressure_hpa", "mean"),
        rainfall_total=("rainfall_hourly_mm", lambda s: s.sum(min_count=1)),
    )
)

# Apply daily PM2.5 completeness rule.
insufficient_mean = daily_air["pm25_valid_hours"] < MIN_VALID_PM25_HOURS_FOR_DAILY_MEAN
daily_air.loc[insufficient_mean, ["pm25_mean", "pm25_max"]] = np.nan

# Daily wind direction reconstructed from vector mean.
daily_air["wind_direction_deg"] = (
    np.degrees(np.arctan2(daily_air["wind_sin"], daily_air["wind_cos"])) + 360.0
) % 360.0
daily_air.loc[daily_air[["wind_sin", "wind_cos"]].isna().all(axis=1), "wind_direction_deg"] = np.nan

# Elevated-hours count from the cleaned hourly PM2.5 data.
hourly_counts = (
    epa.assign(elevated=epa["pm25_ug_m3"].gt(PM25_THRESHOLD))
    .groupby(["station", "date"])
    .agg(
        pm25_valid_hours_for_count=("pm25_ug_m3", "count"),
        elevated_hours_today=("elevated", "sum"),
    )
    .reset_index()
)
hourly_counts.loc[
    hourly_counts["pm25_valid_hours_for_count"] < MIN_VALID_PM25_HOURS_FOR_COUNT,
    "elevated_hours_today"
] = np.nan

daily_air = daily_air.merge(hourly_counts, on=["station", "date"], how="left")
daily_air["exceedance_today"] = np.where(
    daily_air["pm25_mean"].notna(),
    (daily_air["pm25_mean"] > PM25_THRESHOLD).astype(int),
    np.nan,
)

assert not daily_air.duplicated(["station", "date"]).any()
print("Daily station rows:", len(daily_air))
display(daily_air.head())
save_table(daily_air, "epa_daily_station_cleaned.csv")

## 6. Air-quality lag and rolling features

All lags are calculated **within station**. Rolling windows end on day *t*, which is allowed because the target is day *t+1*. The current day's PM₂.₅ is also retained for the persistence baseline and next-day prediction.

In [ ]:
daily_air = daily_air.sort_values(["station", "date"]).copy()

for col in ["pm25_mean", "pm10_mean"]:
    for lag in [1, 2, 3]:
        daily_air[f"{col}_lag{lag}"] = daily_air.groupby("station")[col].shift(lag)
    for window in [3, 7]:
        daily_air[f"{col}_rolling_{window}d"] = (
            daily_air.groupby("station")[col]
            .transform(lambda s: s.rolling(window=window, min_periods=2).mean())
        )

# Cyclic season features.
daily_air["month"] = daily_air["date"].dt.month
daily_air["day_of_year"] = daily_air["date"].dt.dayofyear
daily_air["month_sin"] = np.sin(2 * np.pi * daily_air["month"] / 12.0)
daily_air["month_cos"] = np.cos(2 * np.pi * daily_air["month"] / 12.0)
daily_air["doy_sin"] = np.sin(2 * np.pi * daily_air["day_of_year"] / 365.25)
daily_air["doy_cos"] = np.cos(2 * np.pi * daily_air["day_of_year"] / 365.25)

display(daily_air.head(8))

# Part B — NASA FIRMS S-NPP VIIRS

## 7. File-level audit and memory-safe Darwin pre-filter

The archive is read **once** in chunks. During that same pass the notebook records file-level coverage/quality evidence and keeps only detections close enough to Darwin to matter for later station-specific 500 km features.

This catches:
- misleading filename-year labels;
- overlapping archive periods;
- invalid coordinates/dates;
- missing or negative FRP;
- confidence/day-night coverage.

The pre-filter uses 520 km around a Darwin reference point; the extra 20 km prevents edge losses before exact distances to Palmerston, Winnellie and Stokes Hill are calculated.

In [ ]:
FIRMS_USECOLS = [
    "latitude", "longitude", "brightness",
    "acq_date", "acq_time", "satellite", "instrument",
    "confidence", "frp", "daynight",
]

filtered_chunks = []
fire_audit_rows = []

raw_rows_seen = 0
invalid_coord_rows = 0
invalid_frp_rows = 0
outside_prefilter_rows = 0

for path in FIRMS_PATHS:
    print("Auditing and pre-filtering:", path.name)

    row_count = 0
    min_date = None
    max_date = None
    file_invalid_coord = 0
    file_missing_frp = 0
    file_negative_frp = 0
    confidence_values = set()
    daynight_values = set()

    for chunk in pd.read_csv(path, usecols=FIRMS_USECOLS, chunksize=250_000):
        row_count += len(chunk)
        raw_rows_seen += len(chunk)

        chunk["acq_date"] = pd.to_datetime(chunk["acq_date"], errors="coerce")
        for col in ["latitude", "longitude", "frp", "brightness"]:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

        cmin = chunk["acq_date"].min()
        cmax = chunk["acq_date"].max()
        if pd.notna(cmin):
            min_date = cmin if min_date is None or cmin < min_date else min_date
        if pd.notna(cmax):
            max_date = cmax if max_date is None or cmax > max_date else max_date

        confidence_values.update(chunk["confidence"].dropna().astype(str).str.lower().unique())
        daynight_values.update(chunk["daynight"].dropna().astype(str).str.upper().unique())

        valid_coord = (
            chunk["latitude"].between(-90, 90) &
            chunk["longitude"].between(-180, 180) &
            chunk["acq_date"].notna()
        )
        bad_coord_n = int((~valid_coord).sum())
        file_invalid_coord += bad_coord_n
        invalid_coord_rows += bad_coord_n

        file_missing_frp += int(chunk["frp"].isna().sum())

        chunk = chunk.loc[valid_coord].copy()

        bad_frp = chunk["frp"].notna() & chunk["frp"].lt(0)
        bad_frp_n = int(bad_frp.sum())
        file_negative_frp += bad_frp_n
        invalid_frp_rows += bad_frp_n
        chunk.loc[bad_frp, "frp"] = np.nan

        chunk["distance_to_darwin_ref_km"] = haversine_km(
            chunk["latitude"], chunk["longitude"],
            DARWIN_REFERENCE_LAT, DARWIN_REFERENCE_LON
        )
        keep = chunk["distance_to_darwin_ref_km"] <= DARWIN_PREFILTER_KM
        outside_prefilter_rows += int((~keep).sum())
        chunk = chunk.loc[keep].copy()

        # Category dtypes substantially reduce memory for repeated string labels.
        for cat_col in ["satellite", "instrument", "confidence", "daynight"]:
            chunk[cat_col] = chunk[cat_col].astype("category")

        filtered_chunks.append(chunk)

    claimed_year = parse_year_from_name(path)
    actual_years = (
        list(range(min_date.year, max_date.year + 1))
        if min_date is not None and max_date is not None else []
    )

    fire_audit_rows.append({
        "file": path.name,
        "claimed_year": claimed_year,
        "rows": row_count,
        "min_date": min_date,
        "max_date": max_date,
        "actual_years_spanned": ",".join(map(str, actual_years)),
        "filename_year_matches_actual_span": (
            len(actual_years) == 1 and actual_years[0] == claimed_year
        ),
        "invalid_coordinate_or_date_rows": file_invalid_coord,
        "missing_frp_rows": file_missing_frp,
        "negative_frp_rows": file_negative_frp,
        "confidence_values": ",".join(sorted(confidence_values)),
        "daynight_values": ",".join(sorted(daynight_values)),
    })

fire_file_audit = pd.DataFrame(fire_audit_rows).sort_values("min_date")
display(fire_file_audit)
save_table(fire_file_audit, "firms_file_audit.csv")

firms_prefiltered = pd.concat(filtered_chunks, ignore_index=True)
del filtered_chunks

print(f"Rows retained by Darwin {DARWIN_PREFILTER_KM:.0f} km pre-filter: {len(firms_prefiltered):,}")
print(f"Approximate pre-filtered memory: {firms_prefiltered.memory_usage(deep=True).sum()/1e6:,.1f} MB")

In [ ]:
# Date-range overlap and proposal-scope coverage checks.
overlap_rows = []
audit_records = fire_file_audit.to_dict("records")

for i in range(len(audit_records)):
    for j in range(i + 1, len(audit_records)):
        a, b = audit_records[i], audit_records[j]
        overlap_start = max(pd.Timestamp(a["min_date"]), pd.Timestamp(b["min_date"]))
        overlap_end = min(pd.Timestamp(a["max_date"]), pd.Timestamp(b["max_date"]))
        if overlap_start <= overlap_end:
            overlap_rows.append({
                "file_a": a["file"],
                "file_b": b["file"],
                "overlap_start": overlap_start,
                "overlap_end": overlap_end,
            })

file_overlap_report = pd.DataFrame(overlap_rows)
if file_overlap_report.empty:
    print("No overlapping FIRMS file date ranges detected.")
else:
    print("WARNING: overlapping FIRMS file date ranges detected.")
    display(file_overlap_report)

save_table(file_overlap_report, "firms_file_date_overlap_audit.csv")

audit_years = set()
for row in fire_file_audit.itertuples(index=False):
    if pd.notna(row.min_date) and pd.notna(row.max_date):
        audit_years.update(range(pd.Timestamp(row.min_date).year, pd.Timestamp(row.max_date).year + 1))

missing_project_fire_years = sorted(set(PROJECT_YEARS) - audit_years)
if missing_project_fire_years:
    print(
        "COVERAGE WARNING: no supplied FIRMS archive coverage was detected for project year(s):",
        missing_project_fire_years
    )
    print(
        "These years will NOT be interpreted as zero-fire years. "
        "Add the missing archive(s) if the full 2018–2024 scope is required."
    )
else:
    print("FIRMS coverage spans all requested project years:", PROJECT_YEARS)

## 8. Deduplicate and finalise the pre-filtered FIRMS table

Potential overlap is resolved using a detection key based on acquisition date/time, location, satellite/instrument, FRP, confidence and brightness. The source filename is deliberately not part of the key, so the same detection appearing in two archive files can be removed.

In [ ]:
duplicate_key = [
    "latitude", "longitude", "brightness",
    "acq_date", "acq_time", "satellite", "instrument",
    "confidence", "frp", "daynight",
]

duplicates_before = int(
    firms_prefiltered.duplicated(subset=duplicate_key).sum()
)

firms = (
    firms_prefiltered
    .drop_duplicates(subset=duplicate_key, keep="first")
    .copy()
)
del firms_prefiltered

# Re-apply compact categorical dtypes after concatenation/deduplication.
for cat_col in ["satellite", "instrument", "confidence", "daynight"]:
    firms[cat_col] = firms[cat_col].astype("category")

fire_processing_audit = pd.DataFrame([{
    "raw_rows_seen": raw_rows_seen,
    "invalid_coordinate_or_date_rows": invalid_coord_rows,
    "negative_frp_rows_set_nan": invalid_frp_rows,
    "outside_520km_prefilter_rows": outside_prefilter_rows,
    "rows_after_prefilter_before_dedup": len(firms) + duplicates_before,
    "duplicate_rows_removed": duplicates_before,
    "rows_after_dedup": len(firms),
    "min_date": firms["acq_date"].min(),
    "max_date": firms["acq_date"].max(),
    "memory_mb_after_dedup": round(firms.memory_usage(deep=True).sum() / 1e6, 1),
}])
display(fire_processing_audit)
save_table(fire_processing_audit, "firms_processing_audit.csv")

## 9. Station-specific fire feature engineering

The raw FIRMS archive is Darwin-region filtered once, then distances and bearings are calculated separately for each monitoring station. This is more precise than giving all three stations exactly the same proximity feature.

Daily features include:
- counts and FRP sums within 50/100/250/500 km;
- maximum FRP;
- high-confidence count;
- night-fire count;
- nearest-fire distance;
- 8 directional fire-count sectors;
- mean fire-bearing sine/cosine;
- 3-day and 7-day rolling fire activity.

In [ ]:
station_locations = (
    epa.groupby("station")[["latitude", "longitude"]]
    .median()
    .reset_index()
    .sort_values("station")
)
display(station_locations)
save_table(station_locations, "station_locations.csv")

COMPASS_LABELS = np.array(["N", "NE", "E", "SE", "S", "SW", "W", "NW"])

fire_daily_parts = []

for station_row in station_locations.itertuples(index=False):
    station = station_row.station
    lat0 = float(station_row.latitude)
    lon0 = float(station_row.longitude)

    distance = haversine_km(
        firms["latitude"].to_numpy(),
        firms["longitude"].to_numpy(),
        lat0, lon0
    )
    mask = distance <= MAX_FIRE_DISTANCE_KM

    sf = firms.loc[
        mask,
        ["acq_date", "latitude", "longitude", "frp", "confidence", "daynight"]
    ].copy()
    sf["distance_km"] = distance[mask]
    sf["bearing_deg"] = initial_bearing_deg(
        lat0, lon0, sf["latitude"].to_numpy(), sf["longitude"].to_numpy()
    )
    sf["bearing_sin"] = np.sin(np.radians(sf["bearing_deg"]))
    sf["bearing_cos"] = np.cos(np.radians(sf["bearing_deg"]))

    # 8 compass sectors: N is centred on 0°.
    sector_idx = ((sf["bearing_deg"] + 22.5) // 45).astype(int) % 8
    sf["direction_sector"] = COMPASS_LABELS[sector_idx]

    grouped = sf.groupby("acq_date")
    base = pd.DataFrame(index=grouped.size().index)
    base["fire_count_500km"] = grouped.size().astype(int)
    base["frp_sum_500km"] = grouped["frp"].sum(min_count=1)
    base["frp_max_500km"] = grouped["frp"].max()
    base["high_conf_fire_count_500km"] = grouped["confidence"].apply(lambda s: s.eq("h").sum())
    base["night_fire_count_500km"] = grouped["daynight"].apply(lambda s: s.eq("N").sum())
    base["nearest_fire_distance_km"] = grouped["distance_km"].min()
    base["fire_bearing_sin_mean"] = grouped["bearing_sin"].mean()
    base["fire_bearing_cos_mean"] = grouped["bearing_cos"].mean()

    for band in DISTANCE_BANDS_KM:
        within = sf["distance_km"] <= band
        band_group = sf.loc[within].groupby("acq_date")
        base[f"fire_count_{band}km"] = band_group.size()
        base[f"frp_sum_{band}km"] = band_group["frp"].sum(min_count=1)

    direction_counts = pd.crosstab(sf["acq_date"], sf["direction_sector"])
    for label in COMPASS_LABELS:
        base[f"fire_count_dir_{label}"] = direction_counts.get(label, 0)

    base = base.reset_index(names="date")
    base["station"] = station
    fire_daily_parts.append(base)

fire_daily = pd.concat(fire_daily_parts, ignore_index=True)
fire_daily["date"] = pd.to_datetime(fire_daily["date"])

# Reindex to a complete station-day calendar inside confirmed archive coverage.
fire_min_date = firms["acq_date"].min().normalize()
fire_max_date = firms["acq_date"].max().normalize()
covered_fire_years = sorted(firms["acq_date"].dt.year.unique())

full_index = pd.MultiIndex.from_product(
    [
        station_locations["station"].tolist(),
        pd.date_range(fire_min_date, fire_max_date, freq="D")
    ],
    names=["station", "date"]
)
fire_daily = (
    fire_daily.set_index(["station", "date"])
    .reindex(full_index)
    .reset_index()
)

# Zero is meaningful for counts/sums only when the year has supplied FIRMS coverage.
zero_feature_cols = [
    c for c in fire_daily.columns
    if c.startswith("fire_count_") or c.startswith("frp_sum_")
]
covered_mask = fire_daily["date"].dt.year.isin(covered_fire_years)
fire_daily.loc[covered_mask, zero_feature_cols] = (
    fire_daily.loc[covered_mask, zero_feature_cols].fillna(0)
)

fire_daily = fire_daily.sort_values(["station", "date"]).copy()

for col in ["fire_count_500km", "frp_sum_500km", "high_conf_fire_count_500km", "night_fire_count_500km"]:
    for window in [3, 7]:
        fire_daily[f"{col}_rolling_{window}d"] = (
            fire_daily.groupby("station")[col]
            .transform(lambda s: s.rolling(window, min_periods=1).sum())
        )

print("Confirmed FIRMS date coverage:", fire_min_date.date(), "to", fire_max_date.date())
print("Years present:", covered_fire_years)
print("Station-day fire rows:", len(fire_daily))
display(fire_daily.head())
save_table(fire_daily, "firms_daily_station_features.csv")

# Part C — Merged leakage-safe modelling table

## 10. Merge air/weather and fire features

Fire count/sum zeros are only filled inside years for which actual FIRMS archive coverage is present. Missing archive years are excluded from modelling rather than being mistaken for zero-fire periods.

In [ ]:
model_table = daily_air.merge(
    fire_daily,
    on=["station", "date"],
    how="left",
    validate="one_to_one",
)

model_table["year"] = model_table["date"].dt.year

# Only use station-days whose year has FIRMS archive evidence.
model_table = model_table.loc[
    model_table["year"].isin(covered_fire_years)
].copy()

# Mean fire bearing from vector components.
model_table["fire_bearing_mean_deg"] = (
    np.degrees(
        np.arctan2(
            model_table["fire_bearing_sin_mean"],
            model_table["fire_bearing_cos_mean"]
        )
    ) + 360.0
) % 360.0

# Meteorological wind direction is conventionally the direction FROM which wind blows.
# Alignment near +1 means the mean fire bearing lies in a similar direction from the station.
angle_difference_rad = np.radians(
    model_table["wind_direction_deg"] - model_table["fire_bearing_mean_deg"]
)
model_table["wind_fire_direction_alignment"] = np.cos(angle_difference_rad)
model_table.loc[
    model_table[["wind_direction_deg", "fire_bearing_mean_deg"]].isna().any(axis=1),
    "wind_fire_direction_alignment"
] = np.nan

assert not model_table.duplicated(["station", "date"]).any()
print("Merged station-day rows:", len(model_table))
print("Modelling years after confirmed fire coverage:", sorted(model_table["year"].unique()))
display(model_table.head())

## 11. Create tomorrow targets and enforce day-to-day continuity

Targets are generated by station with `shift(-1)`, then checked to ensure that the next record is exactly one calendar day later. This prevents accidental target jumps if a date is missing.

In [ ]:
model_table = model_table.sort_values(["station", "date"]).copy()

next_date = model_table.groupby("station")["date"].shift(-1)
next_pm25 = model_table.groupby("station")["pm25_mean"].shift(-1)
next_elevated_hours = model_table.groupby("station")["elevated_hours_today"].shift(-1)

is_next_calendar_day = next_date.eq(model_table["date"] + pd.Timedelta(days=1))

model_table["target_pm25_mean_tomorrow"] = next_pm25.where(is_next_calendar_day)
model_table["target_exceedance_tomorrow"] = np.where(
    is_next_calendar_day & next_pm25.notna(),
    (next_pm25 > PM25_THRESHOLD).astype(int),
    np.nan,
)
model_table["target_elevated_hours_tomorrow"] = next_elevated_hours.where(is_next_calendar_day)

# Sanity checks.
assert model_table["target_exceedance_tomorrow"].dropna().isin([0, 1]).all()
assert model_table["target_elevated_hours_tomorrow"].dropna().between(0, 24).all()

target_summary = pd.DataFrame({
    "target": [
        "target_exceedance_tomorrow",
        "target_pm25_mean_tomorrow",
        "target_elevated_hours_tomorrow",
    ],
    "nonmissing_rows": [
        model_table["target_exceedance_tomorrow"].notna().sum(),
        model_table["target_pm25_mean_tomorrow"].notna().sum(),
        model_table["target_elevated_hours_tomorrow"].notna().sum(),
    ],
})
display(target_summary)
save_table(target_summary, "target_availability_summary.csv")

## 12. Feature list and leakage audit

The feature list is explicit. No target or next-day column is selected by wildcard.  
Assumption: the next-day forecast is issued **after day t's daily summary is available**. If the intended operational forecast time changes, current-day aggregates must be redefined to match information available at that forecast issue time.

In [ ]:
categorical_features = ["station"]

base_numeric_features = [
    "pm25_mean", "pm25_max",
    "pm25_mean_lag1", "pm25_mean_lag2", "pm25_mean_lag3",
    "pm25_mean_rolling_3d", "pm25_mean_rolling_7d",
    "pm10_mean", "pm10_mean_lag1", "pm10_mean_lag2", "pm10_mean_lag3",
    "pm10_mean_rolling_3d", "pm10_mean_rolling_7d",
    "relative_humidity_mean", "air_temperature_mean",
    "wind_speed_mean", "wind_sin", "wind_cos",
    "air_pressure_mean", "rainfall_total",
    "month_sin", "month_cos", "doy_sin", "doy_cos",
    "elevated_hours_today",
]

fire_feature_candidates = [
    c for c in model_table.columns
    if (
        c.startswith("fire_count_") or
        c.startswith("frp_sum_") or
        c in {
            "frp_max_500km",
            "high_conf_fire_count_500km",
            "night_fire_count_500km",
            "nearest_fire_distance_km",
            "fire_bearing_sin_mean",
            "fire_bearing_cos_mean",
            "wind_fire_direction_alignment",
        }
    )
]

numeric_features = list(dict.fromkeys(base_numeric_features + fire_feature_candidates))
numeric_features = [c for c in numeric_features if c in model_table.columns]

feature_columns = categorical_features + numeric_features
target_columns = [
    "target_exceedance_tomorrow",
    "target_pm25_mean_tomorrow",
    "target_elevated_hours_tomorrow",
]

assert not any(c.startswith("target_") for c in feature_columns)
assert "date" not in feature_columns
assert "year" not in feature_columns

leakage_audit = pd.DataFrame({
    "feature": feature_columns,
    "type": ["categorical" if c in categorical_features else "numeric" for c in feature_columns],
})
display(leakage_audit)
save_table(leakage_audit, "model_feature_manifest.csv")

processed_path = PROCESSED_DIR / "fire2air_station_day_modelling_table.csv"
model_table.to_csv(processed_path, index=False)
print("Saved processed modelling table:", processed_path.relative_to(PROJECT_ROOT))
print("Features:", len(feature_columns))

## 13. Optional DuckDB storage snapshot

The proposal architecture included DuckDB. This cell records the implemented daily tables in a local analytical database **if `duckdb` is installed**. CSV outputs remain the portable fallback.

In [ ]:
try:
    import duckdb

    duckdb_path = OUTPUT_ROOT / "fire2air_darwin.duckdb"
    con = duckdb.connect(str(duckdb_path))
    con.register("daily_air_df", daily_air)
    con.register("fire_daily_df", fire_daily)
    con.register("model_table_df", model_table)

    con.execute("CREATE OR REPLACE TABLE daily_air AS SELECT * FROM daily_air_df")
    con.execute("CREATE OR REPLACE TABLE fire_daily AS SELECT * FROM fire_daily_df")
    con.execute("CREATE OR REPLACE TABLE model_table AS SELECT * FROM model_table_df")
    con.close()
    print("DuckDB snapshot saved:", duckdb_path.relative_to(PROJECT_ROOT))
except ImportError:
    print("duckdb is not installed. CSV evidence is already saved.")
    print("Optional: install with `pip install duckdb` in your VS Code environment.")

# Part E — Leakage-safe modelling

## 15. Chronological split and preprocessing

No random train/test split is used.

- Training includes available years up to 2022.
- 2023 is validation/model-selection data.
- 2024 remains untouched until the final evaluation cell.

Numeric missing values are imputed with medians learned from the training data inside a pipeline. Station is one-hot encoded. Standardisation is also learned only from training data.

In [ ]:
def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_preprocessor():
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features),
    ], remainder="drop")

def split_for_target(target_col):
    required = feature_columns + [target_col, "date", "year"]
    data = model_table[required].dropna(subset=[target_col]).copy()

    train = data[data["year"] <= TRAIN_END_YEAR].copy()
    val = data[data["year"] == VALIDATION_YEAR].copy()
    test = data[data["year"] == TEST_YEAR].copy()

    if train.empty or val.empty or test.empty:
        raise ValueError(
            f"Insufficient chronological data for {target_col}. "
            f"Rows: train={len(train)}, val={len(val)}, test={len(test)}"
        )

    if not (train["date"].max() < val["date"].min() < test["date"].min()):
        raise AssertionError("Chronological split failed.")

    X_train, y_train = train[feature_columns], train[target_col]
    X_val, y_val = val[feature_columns], val[target_col]
    X_test, y_test = test[feature_columns], test[target_col]

    return train, val, test, X_train, y_train, X_val, y_val, X_test, y_test

split_summary_rows = []
for target in target_columns:
    tr, va, te, *_ = split_for_target(target)
    split_summary_rows.append({
        "target": target,
        "train_rows": len(tr),
        "train_start": tr["date"].min(),
        "train_end": tr["date"].max(),
        "validation_rows": len(va),
        "validation_year": VALIDATION_YEAR,
        "test_rows": len(te),
        "test_year": TEST_YEAR,
    })

split_summary = pd.DataFrame(split_summary_rows)
display(split_summary)
save_table(split_summary, "chronological_split_summary.csv")